# 猫狗图片分类：用 NumPy 从零实现卷积网络

卷积、池化、交叉熵、Adam 全部手写，训练与评估在同一个 notebook 内完成。

依赖 Python 3.8 以上版本与 NumPy。Matplotlib 可选，只影响画图。数据由程序生成，不下载任何东西。


In [ ]:
# ============ 第 0 步：检查环境 ============
import os
import sys
import time

import numpy as np

print("Python:", sys.version.split()[0], "| NumPy:", np.__version__)

try:
    import matplotlib
    if "ipykernel" not in sys.modules:
        matplotlib.use("Agg")          # 非 notebook 环境只画不弹窗
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

try:
    from PIL import Image
    HAS_PIL = True
except ImportError:
    HAS_PIL = False

IN_JUPYTER = "ipykernel" in sys.modules


def show_fig(fig):
    '''显示在单元格下方；不在 notebook 里就直接关掉，不弹窗口。'''
    if IN_JUPYTER:
        try:
            from IPython.display import display
            display(fig)
        except Exception:
            pass
    plt.close(fig)


# --- 参数 ---
FAST_DEMO = False          # True：64 张图跑 1 个 epoch，两秒验证环境
DATA_MODE = "toy"          # "toy" 程序生成 | "folder" 自己的照片目录
DATA_DIR = "data"
IMG_SIZE = 32
N_TRAIN_PER_CLASS = 1000
N_VAL_PER_CLASS = 300
BATCH_SIZE = 64
EPOCHS = 4
LR = 1e-3
SEED = 0

if FAST_DEMO:
    N_TRAIN_PER_CLASS, N_VAL_PER_CLASS, EPOCHS = 64, 32, 1

os.makedirs(DATA_DIR, exist_ok=True)
print("数据模式:", DATA_MODE, "| 尺寸: %dx%d" % (IMG_SIZE, IMG_SIZE))


---
## 第 1 步 数据

公开数据集体量大，课程改用程序生成两类头像：猫尖耳加横胡须，狗垂耳加短口鼻。样本形状为 (N, 3, 32, 32)，像素取值 0 到 1。

有真实照片时把 `DATA_MODE` 改成 `folder`，照片按 `data/cats_dogs/cat/*.jpg` 与 `data/cats_dogs/dog/*.jpg` 放好，读取需要 Pillow。


In [ ]:
# ============ 1.2 读取图像目录（可选）============
def load_image_folder(root, size=IMG_SIZE):
    '''读取 root/cat/*.jpg 与 root/dog/*.jpg，统一缩放到 size。'''
    if not HAS_PIL:
        raise RuntimeError("读自己的图片需要 Pillow：pip install pillow")
    xs, ys = [], []
    for label, name in enumerate(("cat", "dog")):
        folder = os.path.join(root, name)
        if not os.path.isdir(folder):
            raise FileNotFoundError("缺少目录：" + folder)
        for fn in sorted(os.listdir(folder)):
            if not fn.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
                continue
            with Image.open(os.path.join(folder, fn)) as im:
                im = im.convert("RGB").resize((size, size), Image.LANCZOS)
                xs.append(np.asarray(im, dtype=np.float32) / 255.0)
            ys.append(label)
    X = np.stack(xs).transpose(0, 3, 1, 2)  # (高,宽,通道) -> (通道,高,宽)
    return X, np.array(ys, dtype=np.int64)


def make_toy_faces(kind, n, rng, size=IMG_SIZE):
    '''合成简化版"尖耳猫 / 垂耳狗"，仅用于没网时验证流程。'''
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)

    def tri(a, b, c):
        def side(p, q):
            return (p[0] - q[0]) * (yy - q[1]) - (p[1] - q[1]) * (xx - q[0])

        d1, d2, d3 = side(a, b), side(b, c), side(c, a)
        return ~(((d1 < 0) | (d2 < 0) | (d3 < 0)) & ((d1 > 0) | (d2 > 0) | (d3 > 0)))

    out = np.zeros((n, 3, size, size), np.float32)
    for i in range(n):
        img = rng.normal(0.55, 0.05, (size, size, 3)).astype(np.float32)
        cy, cx = size / 2 + rng.uniform(-1.5, 1.5), size / 2 + rng.uniform(-1.5, 1.5)
        r = rng.uniform(7.0, 8.5)
        color = rng.uniform(0.55, 0.9, 3).astype(np.float32)
        img[((yy - cy) ** 2 + (xx - cx) ** 2) <= r * r] = color
        if kind == "cat":
            for sx in (-1, 1):
                img[tri((cx + sx * r * 0.7, cy - r * 1.15),
                        (cx + sx * r * 0.1, cy - r * 0.5),
                        (cx + sx * r * 1.0, cy - r * 0.3))] = color * 0.85
            for sy in (-2, 0, 2):
                img[int(cy + sy), int(cx - r - 1):int(cx - r + 3)] = 0.95
                img[int(cy + sy), int(cx + r - 2):int(cx + r + 2)] = 0.95
        else:
            for sx in (-1, 1):
                ear = (((xx - (cx + sx * r * 0.95)) / (0.45 * r)) ** 2
                       + ((yy - (cy + r * 0.15)) / (0.95 * r)) ** 2) <= 1
                img[ear] = color * 0.75
            img[((xx - cx) ** 2 + (yy - (cy + r * 0.35)) ** 2) <= (0.22 * r) ** 2] = 0.15
        out[i] = np.clip(img, 0, 1).transpose(2, 0, 1)
    return out


In [ ]:
# ============ 1.3 构造数据 ============
t_start = time.time()

if DATA_MODE == "folder":
    X, y = load_image_folder(os.path.join(DATA_DIR, "cats_dogs"))
    order = np.random.default_rng(SEED).permutation(len(X))
    X, y = X[order], y[order]
    cut = int(len(X) * 0.8)
    Xtr, ytr, Xva, yva = X[:cut], y[:cut], X[cut:], y[cut:]
else:
    rng = np.random.default_rng(SEED)
    Xtr = np.concatenate([make_toy_faces("cat", N_TRAIN_PER_CLASS, rng),
                          make_toy_faces("dog", N_TRAIN_PER_CLASS, rng)])
    ytr = np.repeat([0, 1], N_TRAIN_PER_CLASS)
    Xva = np.concatenate([make_toy_faces("cat", N_VAL_PER_CLASS, rng),
                          make_toy_faces("dog", N_VAL_PER_CLASS, rng)])
    yva = np.repeat([0, 1], N_VAL_PER_CLASS)

print("训练集 X:", Xtr.shape, " y:", ytr.shape)
print("验证集 X:", Xva.shape, " y:", yva.shape)
print("像素范围: %.2f ~ %.2f" % (Xtr.min(), Xtr.max()))
print("训练集 猫 %d 张 狗 %d 张" % ((ytr == 0).sum(), (ytr == 1).sum()))
print("耗时 %.1f 秒" % (time.time() - t_start))

In [ ]:
# ============ 1.4 训练样本 ============
if HAS_MPL:
    rng = np.random.default_rng(0)
    idx = rng.choice(len(Xtr), 8, replace=False)
    fig, axes = plt.subplots(2, 4, figsize=(10, 5))
    for ax, i in zip(axes.ravel(), idx):
        ax.imshow(Xtr[i].transpose(1, 2, 0))     # (C,H,W) -> (H,W,C)
        ax.set_title("cat" if ytr[i] == 0 else "dog")
        ax.axis("off")
    fig.suptitle("training samples")
    plt.tight_layout()
    show_fig(fig)


---
## 第 2 步 预处理

按通道减均值除标准差，让输入围绕零分布。均值与标准差只从训练集统计，验证集沿用同一组数值。


In [ ]:
# ============ 2.1 标准化（统计量只来自训练集）============
mean = Xtr.mean(axis=(0, 2, 3), keepdims=True)  # 每个通道一个数
std = Xtr.std(axis=(0, 2, 3), keepdims=True) + 1e-8

Xtr = (Xtr - mean) / std
Xva = (Xva - mean) / std

print("训练集 通道均值:", np.round(mean.ravel(), 3))
print("训练集 通道标准差:", np.round(std.ravel(), 3))
print("标准化后范围: %.2f ~ %.2f" % (Xtr.min(), Xtr.max()))
print("形状不变:", Xtr.shape)

perm = np.random.default_rng(SEED).permutation(len(Xtr))
Xtr, ytr = Xtr[perm], ytr[perm]
print("打乱后前 10 个标签:", ytr[:10])


def to_image(x):
    '''把标准化后的 (3,H,W) 还原成 0~1 的 (H,W,3)。显示图片前必须做这一步。'''
    img = (x[None, ...] * std + mean)[0]      # 用 (1,C,1,1) 的统计量广播到 (C,H,W)
    return np.clip(img.transpose(1, 2, 0), 0, 1)


print("反标准化自检: 形状", to_image(Xtr[0]).shape, "范围 %.2f~%.2f"
      % (to_image(Xtr[0]).min(), to_image(Xtr[0]).max()))


---
## 第 3 步 卷积在算什么

卷积核是一组权重，在图像上滑动，逐位置相乘再求和。下面用固定核演示竖直边缘的响应。


In [ ]:
# ============ 3.1 手算一次卷积（不用任何库）============
image = np.array([[0, 0, 1, 1, 0],
                  [0, 1, 1, 0, 0],
                  [1, 1, 0, 0, 0],
                  [1, 1, 0, 0, 0],
                  [0, 1, 1, 1, 0]], dtype=np.float32)

kernel = np.array([[1, 0, -1],
                   [1, 0, -1],
                   [1, 0, -1]], dtype=np.float32)  # 专门找"竖直边缘"

out = np.zeros((3, 3), np.float32)
for i in range(3):
    for j in range(3):
        out[i, j] = (image[i:i + 3, j:j + 3] * kernel).sum()

print("图片：\n", image)
print("\n卷积核：\n", kernel)
print("\n输出特征图：\n", out)
print("\n响应最强处对应原图中由暗到亮的竖直边缘。")
print("卷积核是一组可学习权重，其数值由数据在训练中确定。")


In [ ]:
# ============ 3.2 固定核在一张样本上的响应 ============
if HAS_MPL:
    gray = to_image(Xtr[0]).mean(axis=2)        # 三通道压成灰度，便于观察
    kernels = [
        ("vertical edge",
         np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], np.float32)),
        ("horizontal edge",
         np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]], np.float32)),
        ("blur", np.ones((3, 3), np.float32) / 9.0),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(12, 3))
    axes[0].imshow(gray, cmap="gray")
    axes[0].set_title("input (gray)")
    for k, (name, ker) in enumerate(kernels):
        feat = np.zeros((gray.shape[0] - 2, gray.shape[1] - 2), np.float32)
        for i in range(feat.shape[0]):
            for j in range(feat.shape[1]):
                feat[i, j] = (gray[i:i + 3, j:j + 3] * ker).sum()
        axes[k + 1].imshow(feat, cmap="gray")
        axes[k + 1].set_title(name)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    show_fig(fig)


---
## 第 4 步 各层实现

每层两个方向：前向给出输出并缓存中间量，反向把上游梯度换成参数梯度与下游梯度。

卷积若按定义写成四层循环，Python 跑不动。把每个窗口拉平成一行，卷积就变成一次矩阵乘法，即 im2col。


In [ ]:
# ============ 4.1 im2col / col2im ============
def im2col(x, kh, kw, stride=1, pad=0):
    '''(N,C,H,W) -> (N*H_out*W_out, C*kh*kw)，每个卷积窗口变成一行'''
    N, C, H, W = x.shape
    H_out = (H + 2 * pad - kh) // stride + 1
    W_out = (W + 2 * pad - kw) // stride + 1
    if pad:
        x = np.pad(x, ((0, 0), (0, 0), (pad, pad), (pad, pad)))
    cols = np.empty((N, C, kh, kw, H_out, W_out), dtype=x.dtype)
    for i in range(kh):
        i_end = i + stride * H_out
        for j in range(kw):
            j_end = j + stride * W_out
            cols[:, :, i, j, :, :] = x[:, :, i:i_end:stride, j:j_end:stride]
    return cols.transpose(0, 4, 5, 1, 2, 3).reshape(N * H_out * W_out, C * kh * kw), H_out, W_out


def col2im(cols, x_shape, kh, kw, stride=1, pad=0):
    '''im2col 的逆操作：把每行梯度加回原位置（重叠处累加）'''
    N, C, H, W = x_shape
    H_out = (H + 2 * pad - kh) // stride + 1
    W_out = (W + 2 * pad - kw) // stride + 1
    cols = cols.reshape(N, H_out, W_out, C, kh, kw).transpose(0, 3, 4, 5, 1, 2)
    buf = np.zeros((N, C, H + 2 * pad, W + 2 * pad), dtype=cols.dtype)
    for i in range(kh):
        for j in range(kw):
            buf[:, :, i:i + stride * H_out:stride,
                       j:j + stride * W_out:stride] += cols[:, :, i, j]
    if pad:
        buf = buf[:, :, pad:pad + H, pad:pad + W]
    return buf


_t = np.random.default_rng(0).normal(size=(2, 3, 8, 8)).astype(np.float32)
_c, _ho, _wo = im2col(_t, 3, 3, 1, 1)
print("im2col:", _t.shape, "->", _c.shape, "  H_out=%d W_out=%d" % (_ho, _wo))
print("col2im:", col2im(_c, _t.shape, 3, 3, 1, 1).shape)
print("行数 = 样本数 2 x 每图窗口 %d = %d；每行 %d 个数（3 通道 x 3x3）"
      % (_ho * _wo, _c.shape[0], _c.shape[1]))


In [ ]:
# ============ 4.2 卷积层 ============
def conv_forward(x, w, b, stride=1, pad=0):
    '''x:(N,C,H,W)  w:(K,C,kh,kw)  b:(K,)  ->  out:(N,K,H_out,W_out) + cache'''
    K, C, kh, kw = w.shape
    w_flat = w.reshape(K, -1)
    cols, H_out, W_out = im2col(x, kh, kw, stride, pad)
    out = cols @ w_flat.T + b                    # (N*Ho*Wo, K)
    out = out.reshape(x.shape[0], H_out, W_out, K).transpose(0, 3, 1, 2)
    return out, (cols, x.shape, w.shape, w_flat, stride, pad)


def conv_backward(dout, cache):
    '''上游梯度 dout -> (dx, dw, db)'''
    cols, x_shape, w_shape, w_flat, stride, pad = cache
    K, C, kh, kw = w_shape
    N, _, H_out, W_out = dout.shape
    dout_flat = dout.transpose(0, 2, 3, 1).reshape(N * H_out * W_out, K)
    db = dout_flat.sum(axis=0)
    dw = (dout_flat.T @ cols).reshape(w_shape)
    dcols = dout_flat @ w_flat
    dx = col2im(dcols, x_shape, kh, kw, stride, pad)
    return dx, dw, db


# 自测
_x = np.random.default_rng(1).normal(size=(2, 3, 8, 8)).astype(np.float32)
_w = np.random.default_rng(2).normal(size=(4, 3, 3, 3)).astype(np.float32) * 0.1
_b = np.zeros(4, np.float32)
_y, _cache = conv_forward(_x, _w, _b, pad=1)
_dx, _dw, _db = conv_backward(np.ones_like(_y), _cache)
print("输出:", _y.shape, "（pad=1 保持尺寸）")
print("dx:", _dx.shape, "dw:", _dw.shape, "db:", _db.shape)


In [ ]:
# ============ 4.3 最大池化（把 2x2 压成 1 个数，只保留最大的）============
def maxpool_forward(x, size=2):
    N, C, H, W = x.shape
    assert H % size == 0 and W % size == 0, "本课里 32/2/2 都能整除"
    H_out, W_out = H // size, W // size
    blocks = x.reshape(N, C, H_out, size, W_out, size).transpose(0, 1, 2, 4, 3, 5)
    blocks = blocks.reshape(N, C, H_out, W_out, size * size)
    idx = blocks.argmax(axis=-1)                 # 记住最大值是窗口里第几个
    out = np.take_along_axis(blocks, idx[..., None], axis=-1)[..., 0]
    return out, (idx, x.shape, size)


def maxpool_backward(dout, cache):
    idx, x_shape, size = cache
    N, C, H, W = x_shape
    H_out, W_out = H // size, W // size
    dblocks = np.zeros((N, C, H_out, W_out, size * size), dtype=dout.dtype)
    np.put_along_axis(dblocks, idx[..., None], dout[..., None], axis=-1)
    dx = dblocks.reshape(N, C, H_out, W_out, size, size).transpose(0, 1, 2, 4, 3, 5)
    return dx.reshape(N, C, H, W)


_x = np.random.default_rng(3).normal(size=(2, 3, 8, 8)).astype(np.float32)
_p, _pc = maxpool_forward(_x, 2)
print("池化:", _x.shape, "->", _p.shape)
print("反传:", maxpool_backward(np.ones_like(_p), _pc).shape)
print("池化不改变通道数，只把空间尺寸减半")


In [ ]:
# ============ 4.4 全连接层 + 交叉熵损失 ============
def softmax_cross_entropy(logits, y):
    '''logits:(N,K)  y:(N,) 整数标签 -> (loss, dlogits)

    先减最大值再 exp，防止数值溢出（这是一个很常见的面试题）。
    '''
    z = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(z)
    probs = exp / exp.sum(axis=1, keepdims=True)
    n = len(y)
    loss = -np.log(probs[np.arange(n), y] + 1e-12).mean()
    dlogits = probs.copy()
    dlogits[np.arange(n), y] -= 1.0              # 这个简洁的梯度是 softmax+CE 的经典结果
    return loss, dlogits / n


_logits = np.array([[2.0, 0.5], [0.1, 1.2]], np.float32)
_y = np.array([0, 1])
_loss, _dl = softmax_cross_entropy(_logits, _y)
print("loss = %.4f（完全预测对的极限是 0）" % _loss)
print("dlogits =\n", np.round(_dl, 4))
print("两行元素之和为零，这是 softmax 与交叉熵复合后的性质")


In [ ]:
# ============ 4.5 Adam 优化器（比 SGD 好用得多）============
class Adam:
    '''一阶动量 + 二阶动量，能自动调整每个参数的学习率。'''

    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps = lr, beta1, beta2, eps
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.t = 0

    def step(self, params, grads):
        self.t += 1
        for k in params:
            g = grads[k]
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * g
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * (g * g)
            m_hat = self.m[k] / (1 - self.b1 ** self.t)   # 偏差修正
            v_hat = self.v[k] / (1 - self.b2 ** self.t)
            params[k] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


print("Adam 实现完成。")


---
## 第 5 步 网络与梯度检验

两个卷积层加两个池化层，再接两层全连接，参数约十三万。

反向传播写错不会报错，损失照样下降。用数值梯度对比解析梯度：给参数加一个极小增量，看两者的相对误差。ReLU 与最大池化在零点不可导，个别参数有千分之一偏差属正常。


In [ ]:
# ============ 5.1 网络本体 ============
class SmallCNN:
    def __init__(self, seed=0):
        rng = np.random.default_rng(seed)
        # He 初始化：方差 2/fan_in，配合 ReLU 效果好
        self.p = {
            "w1": rng.normal(0, np.sqrt(2 / (3 * 9)), (8, 3, 3, 3)).astype(np.float32),
            "b1": np.zeros(8, np.float32),
            "w2": rng.normal(0, np.sqrt(2 / (8 * 9)), (16, 8, 3, 3)).astype(np.float32),
            "b2": np.zeros(16, np.float32),
            "w3": rng.normal(0, np.sqrt(2 / 1024), (1024, 128)).astype(np.float32),
            "b3": np.zeros(128, np.float32),
            "w4": rng.normal(0, np.sqrt(2 / 128), (128, 2)).astype(np.float32),
            "b4": np.zeros(2, np.float32),
        }

    def forward(self, x):
        c1, self._k1 = conv_forward(x, self.p["w1"], self.p["b1"], pad=1)
        r1 = np.maximum(c1, 0)
        p1, self._k2 = maxpool_forward(r1, 2)
        c2, self._k3 = conv_forward(p1, self.p["w2"], self.p["b2"], pad=1)
        r2 = np.maximum(c2, 0)
        p2, self._k4 = maxpool_forward(r2, 2)
        flat = p2.reshape(len(p2), -1)
        h = flat @ self.p["w3"] + self.p["b3"]
        rh = np.maximum(h, 0)
        logits = rh @ self.p["w4"] + self.p["b4"]
        self._cache = (c1, c2, p2, flat, h, rh)   # 反向传播要用的中间结果
        return logits

    def loss_and_grad(self, x, y):
        '''前向 + 反向，返回 (loss, 参数字典的梯度, logits)'''
        logits = self.forward(x)
        loss, dlogits = softmax_cross_entropy(logits, y)
        c1, c2, p2, flat, h, rh = self._cache

        g = {}
        g["w4"] = rh.T @ dlogits                  # 全连接层梯度
        g["b4"] = dlogits.sum(axis=0)
        dh = (dlogits @ self.p["w4"].T) * (h > 0)  # ReLU 的导数：正数处为 1
        g["w3"] = flat.T @ dh
        g["b3"] = dh.sum(axis=0)

        dp2 = (dh @ self.p["w3"].T).reshape(p2.shape)
        dr2 = maxpool_backward(dp2, self._k4) * (c2 > 0)
        dp1, g["w2"], g["b2"] = conv_backward(dr2, self._k3)
        dr1 = maxpool_backward(dp1, self._k2) * (c1 > 0)
        _, g["w1"], g["b1"] = conv_backward(dr1, self._k1)
        return loss, g, logits

    def n_params(self):
        return sum(v.size for v in self.p.values())


model = SmallCNN(SEED)
print("参数总量：%d（约 %.1f 万）" % (model.n_params(), model.n_params() / 1e4))
for k, v in model.p.items():
    print("  %-3s %-16s %d 个" % (k, str(v.shape), v.size))


In [ ]:
# ============ 5.2 形状自检：喂一个 batch 看输出对不对 ============
_xb, _yb = Xtr[:BATCH_SIZE], ytr[:BATCH_SIZE]
_logits = model.forward(_xb)
print("输入:", _xb.shape, "-> 输出:", _logits.shape, "（每张图 2 个分数）")
_loss, _g, _ = model.loss_and_grad(_xb, _yb)
print("初始 loss: %.4f（二分类随机猜是 ln2 = 0.693）" % _loss)
print("梯度键:", sorted(_g.keys()))
print("每个梯度的形状都应与参数一致:", all(_g[k].shape == model.p[k].shape for k in _g))


In [ ]:
# ============ 6.1 数值梯度 vs 反向传播 ============
def numeric_grad(f, arr, index, eps=1e-5):
    old = arr[index]
    arr[index] = old + eps
    f_plus = f()
    arr[index] = old - eps
    f_minus = f()
    arr[index] = old
    return (f_plus - f_minus) / (2 * eps)


def biggest_index(g):
    '''挑梯度最大的位置来检查，比随便挑一个更有意义'''
    return np.unravel_index(int(np.abs(g).argmax()), g.shape)


model = SmallCNN(SEED)
# 用 float64 做检查：float32 的舍入误差会掩盖真正的实现错误
model.p = {k: v.astype(np.float64) for k, v in model.p.items()}
xb, yb = Xtr[:4].astype(np.float64), ytr[:4]
loss, grads, _ = model.loss_and_grad(xb, yb)


def loss_fn():
    return model.loss_and_grad(xb, yb)[0]


print("loss = %.6f" % loss)
print("%-18s %15s %15s %10s" % ("参数(梯度最大处)", "数值梯度", "解析梯度", "相对误差"))
print("-" * 62)
worst = 0.0
for name in ("w1", "w2", "w3", "w4", "b1", "b4"):
    idx = biggest_index(grads[name])
    num = numeric_grad(loss_fn, model.p[name], idx)
    ana = grads[name][idx]
    rel = abs(num - ana) / max(1e-12, abs(num) + abs(ana))
    worst = max(worst, rel)
    print("%-18s %15.8f %15.8f %10.2e" % ("%s%s" % (name, idx), num, ana, rel))

print("\n最大相对误差 %.2e" % worst)
if worst < 1e-2:
    print("误差与解析梯度同量级。")
    print("个别参数偏差在千分之一量级属正常，原因是 ReLU 与最大池化在零点附近不可导。")
else:
    print("误差超出正常范围，检查对应层的反向传播，重点看转置、填充与累加顺序。")


---
## 第 6 步 训练

循环就三步：取批次，算损失与梯度，更新参数。优化器用 Adam。


In [ ]:
# ============ 7.1 工具函数 ============
def iterate_batches(X, y, batch, rng, shuffle=True):
    idx = rng.permutation(len(X)) if shuffle else np.arange(len(X))
    for s in range(0, len(idx) - batch + 1, batch):
        sel = idx[s:s + batch]
        yield X[sel], y[sel]


def predict(model, X, batch=256):
    out = []
    for i in range(0, len(X), batch):
        out.append(model.forward(X[i:i + batch]).argmax(axis=1))
    return np.concatenate(out)


def accuracy(model, X, y):
    return float((predict(model, X) == y).mean())


def augment(x, rng, max_shift=2):
    '''数据增强：随机水平翻转 + 随机平移，让模型见过更多"变体"。

    np.roll 会把移出边界的内容绕到另一侧；课堂演示够用，工程实现应改成
    "先填边再裁剪"，否则会引入人为的重影。
    '''
    out = np.empty_like(x)
    for i in range(len(x)):
        img = x[i][:, :, ::-1] if rng.random() < 0.5 else x[i]
        dy, dx = rng.integers(-max_shift, max_shift + 1, 2)
        out[i] = np.roll(np.roll(img, dy, axis=1), dx, axis=2)
    return out


def train(model, Xtr, ytr, Xva, yva, epochs, batch=BATCH_SIZE, lr=LR,
          seed=SEED, augment_fn=None, verbose=True):
    opt = Adam(model.p, lr)
    rng = np.random.default_rng(seed)
    hist = {"loss": [], "train_acc": [], "val_acc": []}
    for ep in range(1, epochs + 1):
        t0 = time.time()
        losses = []
        for xb, yb in iterate_batches(Xtr, ytr, batch, rng):
            if augment_fn is not None:
                xb = augment_fn(xb, rng)
            loss, grads, _ = model.loss_and_grad(xb, yb)
            opt.step(model.p, grads)
            losses.append(loss)
        tr_acc = accuracy(model, Xtr, ytr)
        va_acc = accuracy(model, Xva, yva)
        hist["loss"].append(float(np.mean(losses)))
        hist["train_acc"].append(tr_acc)
        hist["val_acc"].append(va_acc)
        if verbose:
            print("epoch %2d/%d  loss %.4f  训练 %.3f  验证 %.3f  (%.1fs)"
                  % (ep, epochs, hist["loss"][-1], tr_acc, va_acc, time.time() - t0))
    return hist


print("训练与评估函数已定义")


In [ ]:
# ============ 7.2 训练（CPU 上也只要几分钟）============
model = SmallCNN(SEED)
t0 = time.time()
hist = train(model, Xtr, ytr, Xva, yva, EPOCHS)
train_seconds = time.time() - t0
print("\n总训练时间 %.1f 秒" % train_seconds)


In [ ]:
# ============ 7.3 训练曲线 ============
if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
    ax1.plot(hist["loss"], marker="o")
    ax1.set_xlabel("epoch")
    ax1.set_ylabel("train loss")
    ax1.set_title("loss goes down")
    ax1.grid(alpha=0.3)

    ax2.plot(hist["train_acc"], marker="o", label="train")
    ax2.plot(hist["val_acc"], marker="o", label="val")
    ax2.set_xlabel("epoch")
    ax2.set_ylabel("accuracy")
    ax2.set_title("gap = overfitting")
    ax2.legend()
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    show_fig(fig)
else:
    print("loss:", [round(v, 4) for v in hist["loss"]])
    print("val_acc:", [round(v, 4) for v in hist["val_acc"]])


---
## 第 7 步 评估

准确率看整体，混淆矩阵看两类各自错在哪。训练准确率明显高于验证准确率就是过拟合。


In [ ]:
# ============ 8.1 总体准确率 + 混淆矩阵 ============
pred_va = predict(model, Xva)
val_acc = float((pred_va == yva).mean())
print("验证集准确率: %.4f  (%d/%d)" % (val_acc, (pred_va == yva).sum(), len(yva)))

cm = np.zeros((2, 2), dtype=int)
for t, p in zip(yva, pred_va):
    cm[t, p] += 1
print("\n混淆矩阵（行=真实，列=预测）")
print("          预测猫  预测狗")
print("真实猫    %5d   %5d" % (cm[0, 0], cm[0, 1]))
print("真实狗    %5d   %5d" % (cm[1, 0], cm[1, 1]))
print("\n猫的召回率: %.3f   狗的召回率: %.3f"
      % (cm[0, 0] / max(1, cm[0].sum()), cm[1, 1] / max(1, cm[1].sum())))


In [ ]:
# ============ 8.2 九个预测 ============
if HAS_MPL:
    rng = np.random.default_rng(0)
    idx = rng.choice(len(Xva), 9, replace=False)
    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    for ax, i in zip(axes.ravel(), idx):
        ax.imshow(to_image(Xva[i]))              # 显示前还原到 0~1
        ok = pred_va[i] == yva[i]
        ax.set_title("%s->%s %s" % ("cat" if yva[i] == 0 else "dog",
                                    "cat" if pred_va[i] == 0 else "dog",
                                    "ok" if ok else "wrong"),
                     color="green" if ok else "red")
        ax.axis("off")
    plt.tight_layout()
    show_fig(fig)


---
## 第 8 步 改进

翻转与平移做数据增强。把 `RUN_EXPERIMENT` 改成 True 再训练一次，两次结果对比。


In [ ]:
# ============ 9.1 数据增强对比实验 ============
RUN_EXPERIMENT = False

if RUN_EXPERIMENT:
    model_aug = SmallCNN(SEED)
    hist_aug = train(model_aug, Xtr, ytr, Xva, yva, EPOCHS,
                     augment_fn=lambda x, r: augment(x, r, max_shift=2))
    print("\n基线    验证准确率 %.4f" % hist["val_acc"][-1])
    print("数据增强 验证准确率 %.4f" % hist_aug["val_acc"][-1])
else:
    print("把 RUN_EXPERIMENT 改为 True 可再训练一次，用于与基线比较。")


### 小结

卷积提局部纹理，池化带来平移不变性，两层堆叠感受野变大，交叉熵与 Adam 把误差变成可求导的目标。


---
## 思考题

（1）`IMG_SIZE` 改成 16，准确率怎么变。

（2）去掉标准化，前两个 epoch 的 loss 有什么不同。

（3）最大池化换成平均池化，结果变不变。

（4）`EPOCHS` 调到 30，训练准确率接近 1 时验证准确率怎么变。


---
## 什么时候该用框架

网络很深、需要预训练权重、样本十万以上、必须用 GPU，这几种情况直接上框架。本文的作用是先把计算过程看明白。
